# So sánh các phương pháp giải hệ phương trình Hồi quy tuyến tính trong NumPy

Trong notebook này, chúng ta sẽ thực hành tính toán vectơ hệ số $w$ của mô hình Hồi quy tuyến tính nhiều chiều bằng 3 phương pháp khác nhau trong thư viện NumPy:
1. **`np.linalg.inv`**: Nghịch đảo ma trận thông thường (Yêu cầu $X^T X$ khả nghịch).
2. **`np.linalg.pinv`**: Giả nghịch đảo Moore-Penrose (Hoạt động tốt cả khi $X^T X$ không khả nghịch).
3. **`np.linalg.lstsq`**: Giải bài toán bình phương tối thiểu trực tiếp.

*(Lưu ý: Sửa lỗi chính tả từ yêu cầu: sử dụng gói `linalg` của `numpy` chứ không phải `linalf`)*

In [2]:
import numpy as np

## 1. Chuẩn bị dữ liệu mẫu

Ta sẽ tạo ra 2 trường hợp:
* **Trường hợp 1 (Khả nghịch):** Các cột đặc trưng độc lập tuyến tính hoàn toàn.
* **Trường hợp 2 (Không khả nghịch):** Có cột đặc trưng trùng lặp thông tin tuyến tính (đa cộng tuyến hoàn hảo).

In [3]:
# Số lượng mẫu n = 5, số đặc trưng p = 2 (thêm bias nữa là 3 tham số)
# Dữ liệu y thực tế
y = np.array([[10], [15], [20], [25], [30]])

# Trường hợp 1: X_invertible có các cột độc lập tuyến tính
X_invertible = np.array([
    [1, 2, 3],
    [1, 3, 5],
    [1, 4, 2],
    [1, 5, 8],
    [1, 6, 1]
])

# Trường hợp 2: X_singular có cột thứ 3 gấp đôi cột thứ 2 (phụ thuộc tuyến tính hoàn toàn)
X_singular = np.array([
    [1, 2, 4],
    [1, 3, 6],
    [1, 4, 8],
    [1, 5, 10],
    [1, 6, 12]
])

print("Dữ liệu đã sẵn sàng.")

Dữ liệu đã sẵn sàng.


--- 
## 2. Phương pháp 1: Sử dụng `np.linalg.inv` (Nghịch đảo thông thường)

Công thức OLS dạng vectơ cột:
$$w^T = (X^T X)^{-1} X^T y$$

In [4]:
print("--- THỬ NGHIỆM VỚI DỮ LIỆU KHẢ NGHỊCH ---")
try:
    XTX_inv = np.linalg.inv(X_invertible.T @ X_invertible)
    w_inv = XTX_inv @ X_invertible.T @ y
    print("Hệ số w thu được:")
    print(w_inv.T)
except np.linalg.LinAlgError as e:
    print("Lỗi:", e)

print("\n--- THỬ NGHIỆM VỚI DỮ LIỆU SUY BIẾN (KHÔNG KHẢ NGHỊCH) ---")
try:
    XTX_inv_sing = np.linalg.inv(X_singular.T @ X_singular)
    w_inv_sing = XTX_inv_sing @ X_singular.T @ y
    print(w_inv_sing.T)
except np.linalg.LinAlgError as e:
    print("Lỗi:", e, "<- np.linalg.inv bị lỗi vì ma trận không khả nghịch!")

--- THỬ NGHIỆM VỚI DỮ LIỆU KHẢ NGHỊCH ---
Hệ số w thu được:
[[-9.93094496e-14  5.00000000e+00  7.49400542e-16]]

--- THỬ NGHIỆM VỚI DỮ LIỆU SUY BIẾN (KHÔNG KHẢ NGHỊCH) ---
Lỗi: Singular matrix <- np.linalg.inv bị lỗi vì ma trận không khả nghịch!


--- 
## 3. Phương pháp 2: Sử dụng `np.linalg.pinv` (Giả nghịch đảo Moore-Penrose)

Công thức sử dụng giả nghịch đảo của ma trận thiết kế $X$ (ký hiệu $X^+$):
$$w^T = X^+ y$$
Phương pháp này cực kỳ ổn định vì **không cần** tính $(X^T X)^{-1}$ mà tính trực tiếp giả nghịch đảo của $X$ dựa trên phân tích SVD.

In [ ]:
print("--- THỬ NGHIỆM VỚI DỮ LIỆU KHẢ NGHỊCH ---")
w_pinv = np.linalg.pinv(X_invertible) @ y
print("Hệ số w thu được:")
print(w_pinv.T)

print("\n--- THỬ NGHIỆM VỚI DỮ LIỆU SUY BIẾN (KHÔNG KHẢ NGHỊCH) ---")
w_pinv_sing = np.linalg.pinv(X_singular) @ y
print("Hệ số w thu được (vẫn chạy thành công không báo lỗi):")
print(w_pinv_sing.T)

--- 
## 4. Phương pháp 3: Sử dụng `np.linalg.lstsq` (Giải bình phương tối thiểu trực tiếp)

Hàm này tìm nghiệm $w^T$ cho hệ phương trình $X w^T = y$ sao cho $\|y - X w^T\|_2^2$ đạt cực tiểu.
Nó trả về một tuple chứa: nghiệm `w`, tổng bình phương phần dư `residuals`, hạng của ma trận `rank`, và các giá trị suy biến `s`.

In [ ]:
print("--- THỬ NGHIỆM VỚI DỮ LIỆU KHẢ NGHỊCH ---")
# Trả về nghiệm, phần dư, hạng ma trận, các trị số suy biến
w_lstsq, residuals, rank, s = np.linalg.lstsq(X_invertible, y, rcond=None)
print("Hệ số w thu được:")
print(w_lstsq.T)

print("\n--- THỬ NGHIỆM VỚI DỮ LIỆU SUY BIẾN (KHÔNG KHẢ NGHỊCH) ---")
w_lstsq_sing, residuals_sing, rank_sing, s_sing = np.linalg.lstsq(X_singular, y, rcond=None)
print("Hệ số w thu được (vẫn tìm được nghiệm xấp xỉ tốt nhất):")
print(w_lstsq_sing.T)

## 5. Kết luận

| Phương pháp | Ưu điểm | Nhược điểm | Khuyên dùng khi nào? |
| :--- | :--- | :--- | :--- |
| **`np.linalg.inv`** | Dễ hiểu, bám sát lý thuyết OLS nguyên bản. | Bị crash (LinAlgError) ngay lập tức khi ma trận có đa cộng tuyến hoàn hảo. | Chỉ dùng khi chắc chắn dữ liệu sạch và $X^TX$ khả nghịch. |
| **`np.linalg.pinv`** | Rất an toàn, ổn định cao, không bao giờ bị crash. | Tốn tài nguyên tính toán hơn một chút so với nghịch đảo thông thường. | **Khuyên dùng nhiều nhất** khi cần cài đặt công thức toán dạng đóng trong các bài thực hành học máy. |
| **`np.linalg.lstsq`** | Hiệu năng cực kỳ cao, được tối ưu hóa sâu bằng thư viện LAPACK dưới nền. | Trả về nhiều tham số phụ nên cần xử lý kết quả đầu ra. | Khuyên dùng khi viết các thư viện sản phẩm chuyên nghiệp cần tốc độ cao. |

Trường hợp hệ $Ax = b$ có nghiệm (Hệ tương thích)
Điều kiện có nghiệm: $A A^+ b = b$
Nghiệm tổng quát: $$x = A^+ b + (I - A^+ A)z \quad (\text{với } z \text{ tùy ý})$$ (Mọi nghiệm $x$ tính từ công thức trên đều thỏa mãn chính xác $Ax = b$).
Nghiệm có độ dài nhỏ nhất: $x_0 = A^+ b$ (ứng với $z = 0$) là nghiệm có chuẩn $|x|_2$ nhỏ nhất trong tất cả các nghiệm

Hệ phương trình $Ax = b$ và Nghiệm Giả Nghịch Đảo
Trường hợp hệ vô nghiệm: Ta tìm $x$ sao cho sai số $|Ax - b|_2$ nhỏ nhất (Nghiệm bình phương tối thiểu).
Nghiệm tổng quát: $$x = A^+ b + (I - A^+ A)z \quad (\text{với } z \text{ tùy ý})$$

Áp dụng để tính vectơ trọng số $w$
Trong hồi quy ($X w^T = y$):

Ta chỉ cần tìm một nghiệm tối ưu duy nhất (tương ứng với $z = 0$ để độ dài $|w|_2$ nhỏ nhất).
Công thức tính $w$: $$w^T = X^+ y \quad \implies \quad w = y^T (X^+)^T$$